# NB-0 — Setup and smoke test

Verify the environment and prove the pipeline runs end to end **before** spending a
session on the real data. This notebook needs only the `halo-src` dataset.

Expected runtime: 3–8 minutes. If any stage reports FAILED, fix it here — do not proceed.


In [ ]:
# --- HALO bootstrap -------------------------------------------------------------
# Attach these datasets to this notebook before running (right panel -> Add Data):
#   1. Competition: "ieee-fraud-detection"      (accept the rules first)
#   2. Your source dataset: "halo-src"           (the halo/ package, see RUN_GUIDE.md)
#   3. For stages after NB-1: the previous stage's output dataset
import os, shutil, sys, subprocess, time

SRC = "/kaggle/input/halo-src"
if os.path.exists(SRC):
    if os.path.exists("/kaggle/working/halo"):
        shutil.rmtree("/kaggle/working/halo")
    shutil.copytree(os.path.join(SRC, "halo"), "/kaggle/working/halo")
sys.path.insert(0, "/kaggle/working")

# Carry forward checkpoints and results from the previous stage, if one is attached.
for prev in sorted(p for p in os.listdir("/kaggle/input") if p.startswith("halo-stage")):
    for sub in ("checkpoints", "results", "figures"):
        s = f"/kaggle/input/{prev}/{sub}"
        if os.path.isdir(s):
            os.makedirs(f"/kaggle/working/{sub}", exist_ok=True)
            for f in os.listdir(s):
                shutil.copy2(os.path.join(s, f), f"/kaggle/working/{sub}/{f}")

from halo.io import environment_manifest
env = environment_manifest()
print("ENVIRONMENT (observed, not assumed):")
for k, v in env.items():
    print(f"  {k:24s} {v}")


In [ ]:
# --- Smoke test first. Never launch the full grid before this passes. -------------
# Runs the entire pipeline on generated data with known ground truth in a few minutes.
from halo.cli import main
main(["smoke", "--entities", "3000", "--seeds", "0", "1"])
